## LAES optimisation

In [ ]:
# Ensure that python finds the submodules
import sys
sys.path.append("..") # Adds higher directory to python modules path.

# Scientific computing
import numpy as np

#import optimization class
from cryoevap.optimize import Opti_jax

# Visualisation
import matplotlib.pyplot as plt

## Module imports
# Import the storage tank Class
from cryoevap.storage_tanks import Tank

# Import Cryogen class
from cryoevap.cryogens import Cryogen

# Import JAX library
import jax
import jax.numpy as jnp

# Import pandas for data handling
import pandas as pd

# Set JAX to use 64-bit precision
jax.config.update("jax_enable_x64", True)


#### Setup tank and cryogen properties

In [ ]:
# LNG tank properties
Q_roof = 0              # Roof heat ingress / W
T_air  = 18.08 + 273.15  # Temperature of the environment K

V_tank = 4831 # Tank volume / m^3

a   = 0.5
d_i = ((4 * V_tank)/(np.pi * a))**(1/3) # internal diameter / m

# Thickness of the in % of the internal diameter
ST  = 1.02
d_o = d_i * ST # external diameter / m

# Set overall heat transfer coefficient through the walls for liquid and vapour
U_L = 8.72e-2 # W/m2/K
U_V = 8.72e-2 # W/m2/K

# Initial liquid filling / Dimensionless
LF = 0.95

# Specify tank operating pressure
P = 101325*3 # Pa

# # Initialise cryogen
cryogen = Cryogen(name = "nitrogen")
cryogen.set_coolprops(P)

In [ ]:
def estimate_transient_period(a, rq, LF):
    d_i = ((4 * V_tank)/(np.pi * a))**(1/3) # internal diameter / m
    ST  = 1.02
    d_o = d_i * ST # external diameter / m
    H   = V_tank / (0.25 * np.pi * d_i**2)
    q_b = 4 * U_L * H * (T_air - cryogen.T_sat) / (d_o * (1/rq - 1))

    # Initialize large-scale tank
    large_tank = Tank(d_i, d_o, V_tank, LF)
    large_tank.set_HeatTransProps(U_L, U_V, T_air, q_b_fixed = q_b, Q_roof = 0, eta_w = 0.90)
    large_tank.cryogen = cryogen

    # Minimum number of hours to achieve steady state 
    return large_tank.tau


In [ ]:
a  = np.linspace(0.1, 2.5, 500)
a_relation = 1/(a**(1/3))
tau_rq_03_LF95 = estimate_transient_period(a, 0.3, 0.95)
tau_rq_08_LF95 = estimate_transient_period(a, 0.8, 0.95)

tau_rq_03_LF05 = estimate_transient_period(a, 0.3, 0.05)
tau_rq_08_LF05 = estimate_transient_period(a, 0.8, 0.05)


In [ ]:
pd.DataFrame({
    'a': a,
    'tau_rq_03_LF95': tau_rq_03_LF95,
    'tau_rq_08_LF95': tau_rq_08_LF95,
    'tau_rq_03_LF05': tau_rq_03_LF05,
    'tau_rq_08_LF05': tau_rq_08_LF05
}).to_csv('transient_period.csv', index=False)

In [ ]:
# Calculate q_b from thermal aspect ratio
rq1 = 0.7
H   = V_tank / (0.25 * np.pi * d_i**2)
q_b_1 = 4 * U_L * H * (T_air - cryogen.T_sat) / (d_o * (1/rq1 - 1))

rq2 = 0.3 
q_b_2 = 4 * U_L * H * (T_air - cryogen.T_sat) / (d_o * (1/rq2 - 1))

print("Bottom heat flux ingress for r_q = 0.7: ", q_b_1, " W/m^2")
print("Bottom heat flux ingress for r_q = 0.3: ", q_b_2, " W/m^2")

In [ ]:
# Set fixed bottom heat ingress
q_b_fixed = 14.888195

# Initialize large-scale tank
large_tank = Tank(d_i, d_o, V_tank, LF)
large_tank.set_HeatTransProps(U_L, U_V, T_air, q_b_fixed = q_b_fixed, Q_roof = 0, eta_w = 0.90)

# # Set cryogen
large_tank.cryogen = cryogen

# Calculate initial evaporation rate
print("The initial evaporation rate of " + cryogen.name + " is %.1f kg/h" % (large_tank.b_l_dot * 3600))
# comparison of heat flux
print("q_b used: ", q_b_fixed, " q_b free: ", U_L*(T_air - cryogen.T_sat))
print("Q_b used: ", large_tank.Q_b, " Q_b free: ", U_L*(T_air - cryogen.T_sat)*(np.pi*((d_i**2)/4)))

Calculate initial evaporation rate and transient period

In [ ]:
# Calculate initial evaporation rate
print("The initial evaporation rate of " + cryogen.name + " is %.1f kg/h" % (large_tank.b_l_dot * 3600))

# Estimate transient period duration
print("Transient period = %.3f s " % large_tank.tau)

# Minimum number of hours to achieve steady state 
tau_h = (np.floor(large_tank.tau / 3600) + 1)

# Print simulation time of the transient period for short-term storage
print("Simulation time: %.0i h" % tau_h )

# Calculate boil-off rate
BOR = (large_tank.b_l_dot * 24 * 3600) / (large_tank.V * large_tank.LF * large_tank.cryogen.rho_L)
print("BOR = %.3f %%" % (BOR * 100))

#### Simulation setup and execution

In [ ]:
# Define vertical spacing
dz = 0.1

# Calculate number of nodes
n_z = 10 #+ int(np.round(large_tank.l_V/dz, 0))

# Define dimensionless computational grid
large_tank.z_grid = np.linspace(0, 1, n_z)

# Insulated roof
large_tank.U_roof = 0

# Define evaporation time as 12 hours
evap_time = 3600 * 12

# Time step to record data, relevant for plotting integrated quantities
# such as the vapour to liquid heat transfer rate, Q_VL
large_tank.time_interval = 1200

# Time step to plot each vapour temperature profile
large_tank.plot_interval = evap_time/6

# Simulate the evaporation
# large_tank.evaporate(evap_time)

In [ ]:
opti = Opti_jax(large_tank)

In [ ]:
folder    = "../Results/Data/"
optimos   = pd.read_csv(folder + 'LAES_opti_30d_LFs_rq03.csv')
optimo_95 = optimos["Geometric_AR"].iloc[optimos["BOR_LF_0.95"].idxmin()]
optimo_05 = optimos["Geometric_AR"].iloc[optimos["BOR_LF_0.05"].idxmin()]
print("Optimal Geometric AR for LF=0.95: ", optimo_95)
print("Optimal Geometric AR for LF=0.05: ", optimo_05)

In [ ]:
# Exportar perfil de temperatura
opti.time = 3600 * 24 * 30
opti_ar   = 0.430303030303030
sol       = opti.evaporate(opti_ar)

# Convert sol.ys[:,1:] to DataFrame
df_sol = pd.DataFrame(sol.ys[:,1:])
df_sol.insert(0, "Time (s)", sol.ts)

# Export to CSV
folder = "../Results/Data/"
df_sol.to_csv(folder + 'T_V_LF95_30days_rq03.csv', index=False)

In [ ]:
folder = "../Results/Data/"
filename = folder + 'LAES_opti_1y_LFs_rq07.csv'
opti.plot_surface_response_thermal_aspect_ratio_liquid_filling(jnp.linspace(0.2, 4, 100),
                                                               jnp.linspace(0.5, 0.95, 10),
                                                               365*24*3600,
                                                               save=True,
                                                               filename=filename)

In [ ]:
opti.time = 3600 * 24 * 30
LF_array  = jnp.linspace(0.05, 0.95, 200)
def opti_placeholder(LF):
    opti.tank.LF = LF
    opti.params = opti.make_params(opti, opti.tank)
    optimo_ar, min_bor  = opti.optimize_grid_with_refinement(t_final=opti.time,
                                    aspect_ratio_min = 0.2,
                                    aspect_ratio_max = 3.0,
                                    coarse_samples = 1000,
                                    fine_samples = 1000,
                                    verbose = False)
    return [optimo_ar, min_bor, LF]

optimos = [opti_placeholder(LF) for LF in LF_array]

# Split the tuples into separate lists for AR and BOR
optimal_AR, min_bor, LF_values = zip(*optimos)
# Save LF_array, optimal_AR, and min_bor to CSV
df_optimos = pd.DataFrame({
    "LF": np.array(LF_array),
    "Optimal_AR": np.array(optimal_AR),
    "Min_BOR": np.array(min_bor)
})

folder = "../Results/Data/"
df_optimos.to_csv(folder + 'LAES_opti_30d_LFs_rq03_opts.csv', index=False)

#### Visualisation of results

In [ ]:
large_tank.evaporate(12*3600)

#### Vapour temperature

In [ ]:
# Visualise the plot
large_tank.plot_tv(t_unit='h')

Visualise liquid and vapour heat ingresses, $\dot{Q}_{\text{L}}$ and  $\dot{Q}_{\text{V}}$.

The plot also shows the vapour to liquid heat ingress, $\dot{Q}_{VL}$, and  the partition of the vapour heat ingress that is transferred to the interface by the wall directly, $\dot{Q}_{\text{V,w}}$

In [ ]:
large_tank.plot_Q(unit='kW', t_unit='h')

#### Plot liquid volume

In [ ]:
large_tank.plot_V_L()

In [ ]:
V_L = large_tank.data["V_L"]
t   = large_tank.data["Time"]

In [ ]:
(1.0 - V_L[-1] / V_L[0]) * (86400.0 / t[-1])

In [ ]:
large_tank.plot_BOG(unit='kg/s', t_unit='w')

Optional: CSV data export

If evaporation data is intended to be post-processed in another software, it can be exported readily with the help of the Pandas package.

In [ ]:
# Import pandas 
import pandas as pd

In [ ]:
# Create dataframe from dictionary
df_evap = pd.DataFrame.from_dict(large_tank.data)

# Save file to the current working directory
df_evap.to_csv('methane_165000m3.csv')

# Show the first five columns of the dataframe in console
df_evap.head()

In [ ]:
# BOR = (1 - large_tank.data['V_L'][-1]/large_tank.data['V_L'][0])* (86400/large_tank.data['Time'][-1])
BOR = large_tank.BOR()
print("BOR = %.3e %%" % (100 * BOR))

In [ ]:
large_tank.BOR()

In [ ]:
large_tank.data['Time'][-1]

In [ ]:
large_tank.plot_tv_BOG(t_unit='w')

#### References



F. Huerta, V. Vesovic, A realistic vapour phase heat transfer model for the weathering of LNG stored in large tanks, Energy, 174 (2019) 280-291.